In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 9 — A GEOMETRIA DA SEPARAÇÃO: SUPPORT VECTOR MACHINES

> "Em um mundo de pontos, a linha mais forte não é a que passa pelo meio, mas a que se mantém o mais longe possível de todos. A verdadeira força está na margem."
> — Axioma dos Geômetras de Dados

O sucesso do Naive Bayes foi animador, mas sua lógica era de crenças, não de paisagens. Eu não conseguia tirar da cabeça os gráficos da EDA: nuvens de pontos benignos e malignos pairando num espaço de 30 dimensões. Eu queria uma abordagem que pensasse **geometricamente** — que traçasse a fronteira mais limpa possível entre as duas nuvens.

As **Support Vector Machines (SVM)** faziam exatamente isso. Elas não se importavam com o centro das nuvens: eram obcecadas pelas bordas. O SVM busca a "rua" mais larga possível entre as classes, e os pontos que sustentam essa rua são os **vetores de suporte**. Focar nos casos mais difíceis, na fronteira — exatamente onde estava a Helena — ressoou profundamente em mim.

## SVM: A Busca Pela Margem Máxima

O SVM encontra o **hiperplano** que melhor separa as classes, definido como aquele de **margem máxima**: a maior distância possível até os pontos mais próximos de cada classe (os **vetores de suporte**). Como só esses pontos importam, o SVM é robusto e eficiente em memória.

**O truque do kernel.** E se os dados não forem separáveis por uma reta? O SVM os mapeia para um espaço de dimensão mais alta, onde se tornam linearmente separáveis, e encontra ali o hiperplano de margem máxima — sem pagar o custo de calcular essas dimensões explicitamente. Os kernels mais comuns:

**Linear** — para dados já linearmente separáveis.

**RBF (Radial Basis Function)** — o mais flexível; cria fronteiras complexas. Boa primeira escolha quando não se conhece a estrutura dos dados.

O SVM é **sensível à escala**, então o `StandardScaler` é indispensável — encapsulado num *pipeline* para ser ajustado corretamente dentro de cada *fold*.

#### Código 9.1: SVM com Kernels Linear e RBF


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import cross_validate
from oncoclassify_utils import X_train, y_train, recall_maligno

metricas = {"acuracia": "accuracy", "recall_Maligno": recall_maligno}

for kernel in ["linear", "rbf"]:
    pipe = make_pipeline(StandardScaler(), SVC(kernel=kernel, random_state=42))
    r = cross_validate(pipe, X_train, y_train, cv=5, scoring=metricas)
    print(f"SVM {kernel:6s} -> acuracia {r['test_acuracia'].mean():.4f} | "
          f"recall Maligno {r['test_recall_Maligno'].mean():.4f}")

SVM linear -> acuracia 0.9563 | recall Maligno 0.9376
SVM rbf    -> acuracia 0.9750 | recall Maligno 0.9605


Resultados fortes. O kernel **linear** já chega a **95,63%** de acurácia, sugerindo que os dados são, em boa parte, linearmente separáveis. O kernel **RBF**, mais flexível, vai além: **97,50%** de acurácia e, sobretudo, **recall maligno de 96,05%** — o melhor até agora, superando o Naive Bayes (90,4%). Há nuances e curvaturas na fronteira que só o RBF capturou. O SVM se estabelece como candidato forte ao modelo final.

> **⚠️ ATENÇÃO — Os hiperparâmetros C e gamma**
>
> O desempenho do SVM-RBF depende de dois hiperparâmetros:
>
> **C (regularização)** — controla o equilíbrio entre margem larga e classificar todos os pontos. C alto arrisca *overfitting*; C baixo, margem mais larga e mais tolerante.
>
> **gamma** — define o alcance da influência de cada ponto. gamma alto = fronteira mais sinuosa; gamma baixo = fronteira mais suave.
>
> Aqui usamos os valores padrão. No Capítulo 14 vamos otimizá-los sistematicamente.

> **📌 CHECKLIST DO CAPÍTULO 9**
>
> ✅ Entende a intuição geométrica do SVM: hiperplano de **margem máxima**.
>
> ✅ Sabe o que são **vetores de suporte** e o **truque do kernel**.
>
> ✅ Conhece os kernels **linear** e **RBF**.
>
> ✅ Executou o Código 9.1 e viu o RBF superar o linear (**97,5%** vs 95,6%; recall maligno **96,0%** vs 93,8%).
>
> **⚠️ CRÍTICO:** O SVM é sensível à escala e à escolha de C e gamma. Sempre dentro de um *pipeline* com `StandardScaler`.

O SVM-RBF brilhava na tela: 97,5% de acurácia, 96% de recall maligno. Eu havia encontrado uma fronteira forte, traçada com precisão. Sentia-me perto de uma solução robusta.

Mas a jornada pelo arsenal continuava. O SVM era poderoso, porém sua lógica interna, com o kernel RBF, era um tanto opaca. Meu próximo passo seria explorar uma família com filosofia oposta: em vez de uma fronteira geométrica, uma sequência de perguntas simples — a sabedoria das árvores de decisão.
